In [22]:
import os
import json 

NOTEBOOK_DIR= os.getcwd()
BASE_DIR= os.path.join(NOTEBOOK_DIR, "..", "..")
PROCESSED_DATA_DIR= os.path.join(BASE_DIR, "data","processed","pubmedqa")

JSON_FILE= os.path.join(PROCESSED_DATA_DIR,"docs_context.json")

#### PubMedQA

In [23]:
with open(JSON_FILE, "r", encoding="utf-8") as f:
    docs = json.load(f)

texts= [d["text"] for d in docs]

#### MedQA textbooks

### Generative RAG

In [24]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda"
)

In [25]:
embeddings= embed_model.encode(texts, show_progress_bar=True, batch_size= 64)

Batches: 100%|██████████| 6375/6375 [17:26<00:00,  6.09it/s]


In [26]:
import faiss
import numpy as np

embeddings= np.array(embeddings).astype("float16")
dim= embeddings.shape[1]
index= faiss.IndexFlatL2(dim)
index.add(embeddings)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch


model_id = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator1 = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.07s/it]
Device set to use cuda:0


In [38]:
query= 'Is Hirschsprung disease a mendelian or a multifactorial disorder?'


q_embedding= embed_model.encode([query]).astype("float32")

k=3
distances, indices= index.search(q_embedding, k)
retrieved_chunks= [texts[i] for i in indices[0]]

In [39]:
print(retrieved_chunks)

['Hirschsprung disease (HSCR), which may be sporadic or familial, occurs in 1:5000 live births and presents with functional intestinal obstruction secondary to aganglionosis of the hindgut. Germline mutations of the RET proto-oncogene are believed to account for up to 50% of familial cases and up to 30% of isolated cases in most series. However, these series are highly selected for the most obvious and severe cases and large familial aggregations. Population based studies indicate that germline RET mutations account for no more than 3% of isolated HSCR cases. Recently, we and others have noted that specific polymorphic sequence variants, notably A45A (exon 2), are over-represented in isolated HSCR. In order to determine if it is the variant per se, a combination thereof, or another locus in linkage disequilibrium which predisposes to HSCR, we looked for association of RET haplotype(s) and disease in HSCR cases compared to region matched controls. Seven loci across RET were typed and ha

In [40]:
context= "\n\n".join(retrieved_chunks)

prompt = f"""
You are a biomedical expert.

Use ONLY the information in the context to answer the question.


### Context
{context}

### Question
{query}

### Answer
"""

In [ ]:
output1 = generator1(
    prompt,
    max_new_tokens= 30,
    do_sample=False,
    temperature=0,
    return_full_text= False,
)

generated1= str(output1[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
print(generated1)

Hirschsprung disease is a multifactorial disorder.

### Explanation
Hirschsprung disease (HD) is a congenital


In [46]:
from transformers import pipeline

generator2 = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=0
)

Device set to use cuda:0


In [52]:
output2 = generator2(
    prompt,
    max_new_tokens= 60,
    do_sample=False,
    temperature=0)

generated2= str(output2[0]["generated_text"])

In [53]:
print(generated2)

Hirschsprung's disease (HD) is a congenital malformation of the hindgut characterized by the absence of enteric ganglion cells in the submucosal and myenteric plexuses.


### Extractive RAG

In [33]:
from transformers import pipeline

qa = pipeline(
    "question-answering",
    model="ktrapeznikov/biobert_v1.1_pubmed_squad_v2"
)

Some weights of the model checkpoint at ktrapeznikov/biobert_v1.1_pubmed_squad_v2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [43]:
query = 'Is Hirschsprung disease a mendelian or a multifactorial disorder?'


query_embedding = embed_model.encode([query])

scores, indices = index.search(query_embedding, k=5)

retrieved_chunks = [texts[i] for i in indices[0]]



In [44]:
print(retrieved_chunks)

['Hirschsprung disease (HSCR), which may be sporadic or familial, occurs in 1:5000 live births and presents with functional intestinal obstruction secondary to aganglionosis of the hindgut. Germline mutations of the RET proto-oncogene are believed to account for up to 50% of familial cases and up to 30% of isolated cases in most series. However, these series are highly selected for the most obvious and severe cases and large familial aggregations. Population based studies indicate that germline RET mutations account for no more than 3% of isolated HSCR cases. Recently, we and others have noted that specific polymorphic sequence variants, notably A45A (exon 2), are over-represented in isolated HSCR. In order to determine if it is the variant per se, a combination thereof, or another locus in linkage disequilibrium which predisposes to HSCR, we looked for association of RET haplotype(s) and disease in HSCR cases compared to region matched controls. Seven loci across RET were typed and ha

In [45]:
best_answer = ""
best_score = 0

for chunk in retrieved_chunks:

    result = qa(
        question=query,
        context=chunk
    )

    if result["score"] > best_score:
        best_score = result["score"]
        best_answer = result["answer"]

print(best_answer)

multifactorial
